### 3. Perform 𝑘-fold cross-validation from scratch to find the optimal 𝜆 for a Ridge Regression model. Compare your "hand-coded" optimal 𝜆 with the one selected by RidgeCV from Scikit-Learn.

In [7]:
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import RidgeCV

# Generate Synthetic Data
np.random.seed(42)

X = np.random.randn(100, 10)
true_w = np.random.randn(10)
y = X @ true_w + np.random.randn(100) * 2

# Closed Form Ridge Function
def ridge_closed_form(X, y, lam):
    n_features = X.shape[1]
    I = np.eye(n_features)
    return np.linalg.inv(X.T @ X + lam * I) @ X.T @ y

# Manual K-Fold CV Function
def k_fold_ridge(X, y, lambdas, k=5):

    n = len(X)
    indices = np.arange(n)
    np.random.shuffle(indices)

    X = X[indices]
    y = y[indices]

    fold_size = n // k
    best_lambda = None
    best_score = float("inf")

    for lam in lambdas:
        mse_total = 0

        for i in range(k):
            start = i * fold_size
            end = start + fold_size

            X_val = X[start:end]
            y_val = y[start:end]

            X_train = np.concatenate((X[:start], X[end:]), axis=0)
            y_train = np.concatenate((y[:start], y[end:]), axis=0)

            beta = ridge_closed_form(X_train, y_train, lam)
            y_pred = X_val @ beta

            mse_total += mean_squared_error(y_val, y_pred)

        mse_avg = mse_total / k

        if mse_avg < best_score:
            best_score = mse_avg
            best_lambda = lam

    return best_lambda

# Run Manual CV
lambdas = np.logspace(-3, 3, 20)

best_lam_manual = k_fold_ridge(X, y, lambdas)

print("Best Lambda (Manual CV):", best_lam_manual)

# Compare with sklearn RidgeCV
ridge_cv = RidgeCV(alphas=lambdas, cv=5)
ridge_cv.fit(X, y)

print("Best Lambda (RidgeCV):", ridge_cv.alpha_)


Best Lambda (Manual CV): 2.976351441631316
Best Lambda (RidgeCV): 6.158482110660261
